# Module 4: The Model Swarms Algorithm — Deep Dive

**Estimated time: 60 minutes**

This is the core module of the course. We walk through the complete Model Swarms algorithm step by step, connecting each piece to the paper's formulation.

## 4.1 Algorithm Overview

The algorithm has four phases:

```
INITIALIZATION → [VELOCITY UPDATE → WEIGHT UPDATE → CONVERGENCE CHECK] × K
```

The bracketed portion repeats for up to K iterations (default: 200, but patience-based early stopping usually triggers much sooner).

## 4.2 Step 0: Initialization

### Input Requirements
- **n initial experts**: LoRA adapters fine-tuned on different domains (default: 10)
- **Utility function f**: A function that takes a model and returns a scalar score
- **Hyperparameters**: inertia, cognitive/social/repel coefficients, step length, patience

### Population Expansion
The n initial experts are expanded to N particles (default: N=20) through pairwise interpolation with $t \in [0, 2]$ (allowing extrapolation).

### Velocity Initialization

| Mode | How | Intuition |
|------|-----|-----------|
| `zero` | $v_i = 0$ | Particles start stationary |
| `random` | $v_i = x_j - x_i$ (random j) | Each particle points toward a random other particle |
| `best` | $v_i = g - x_i$ | All particles point toward the global best |

The default is `random`, which provides the most diverse initial exploration.

## 4.3 Step 1: Velocity Update (The Core Equation)

This is the heart of Model Swarms:

$$v_i \leftarrow \frac{1}{C} \left[ r_v \cdot \phi_v \cdot v_i + r_p \cdot \phi_p \cdot (p_i - x_i) + r_g \cdot \phi_g \cdot (g - x_i) - r_w \cdot \phi_w \cdot (g_w - x_i) \right]$$

where:
$$C = r_v \cdot \phi_v + r_p \cdot \phi_p + r_g \cdot \phi_g + r_w \cdot \phi_w$$

### Each Component in Detail

| Component | Coefficient | Default | Effect |
|-----------|-------------|---------|--------|
| **Inertia** $r_v \cdot \phi_v \cdot v_i$ | $\phi_v$ | 0.2 | Keep moving in current direction |
| **Cognitive** $r_p \cdot \phi_p \cdot (p_i - x_i)$ | $\phi_p$ | 0.3 | Pull toward personal best |
| **Social** $r_g \cdot \phi_g \cdot (g - x_i)$ | $\phi_g$ | 0.4 | Pull toward global best |
| **Repulsion** $-r_w \cdot \phi_w \cdot (g_w - x_i)$ | $\phi_w$ | 0.1 | Push away from global worst |

The random multipliers $r_v, r_p, r_g, r_w \in [0, 1]$ ensure each particle weighs the components differently at each step, maintaining swarm diversity.

## 4.4 Implementing the Velocity Update

Let's implement the velocity update function and test it:

In [ ]:
import numpy as np
from model_swarms_course.pso import model_swarms_velocity_update

print("Imported model_swarms_velocity_update from src/model_swarms_course/pso.py")


### Testing the Velocity Update

Let's verify with deterministic mode first (no randomness), then explore the stochastic version:

In [ ]:
# Test 1: Deterministic velocity update
v = model_swarms_velocity_update(
    current_velocity=np.array([0.1, -0.05]),
    current_position=np.array([0.3, 0.7]),
    personal_best=np.array([0.4, 0.6]),
    global_best=np.array([0.7, 0.8]),
    global_worst=np.array([0.5, 0.2]),
    use_randomness=False
)
print(f"Deterministic velocity: {v}")

# With randomness — different seeds give different results
for seed in [42, 99]:
    np.random.seed(seed)
    v = model_swarms_velocity_update(
        current_velocity=np.array([0.1, -0.05]),
        current_position=np.array([0.3, 0.7]),
        personal_best=np.array([0.4, 0.6]),
        global_best=np.array([0.7, 0.8]),
        global_worst=np.array([0.5, 0.2]),
        use_randomness=True
    )
    print(f"Random velocity (seed={seed}): {v}")

## 4.5 Step 2: Weight Update

After computing the velocity, the particle moves:

$$x_i \leftarrow x_i + \lambda \cdot v_i$$

where $\lambda$ is the step length (default: starts at 0.7–1.0).

## 4.6 Convergence and Scheduling

### Early Stopping (Patience)
The search terminates when the global best hasn't improved in `patience` iterations (default: 10).

### Step Length Decay
Each iteration: $\lambda \leftarrow \max(\lambda \cdot \phi_\lambda, \lambda_{min})$

| After N iterations | Step length ($\phi_\lambda = 0.95$) |
|---|---|
| 10 | $1.0 \times 0.95^{10} \approx 0.60$ |
| 20 | $1.0 \times 0.95^{20} \approx 0.36$ |
| 30 | $1.0 \times 0.95^{30} \approx 0.21$ |

### Particle Restart
If a particle hasn't improved in `restart_patience * patience` iterations (default: $0.67 \times 10 \approx 7$), it resets to its personal best with zero velocity.

In [ ]:
# Visualize step length decay
import matplotlib.pyplot as plt

iterations = np.arange(50)
decay_rates = [0.80, 0.90, 0.95, 1.00]

fig, ax = plt.subplots(figsize=(10, 5))
for rate in decay_rates:
    step_lengths = np.maximum(rate ** iterations, 0.1)
    ax.plot(iterations, step_lengths, label=f'decay={rate}', linewidth=2)

ax.set_xlabel('Iteration')
ax.set_ylabel('Step Length (λ)')
ax.set_title('Step Length Decay Schedules')
ax.legend()
ax.grid(True, alpha=0.3)
ax.axhline(y=0.1, color='gray', linestyle='--', alpha=0.5, label='minimum')
plt.tight_layout()
plt.show()

## 4.7 The Complete Algorithm as Pseudocode

```
Algorithm: Model Swarms

Input:
  - n initial expert LoRA adapters
  - Utility function f (validation performance)
  - Hyperparameters: φ_v, φ_p, φ_g, φ_w, λ, φ_λ, patience, K

Step 0: Initialize
  Expand n experts to N particles via pairwise interpolation
  Initialize velocities (random mode: v_i = x_random - x_i)
  Evaluate all particles: score_i = f(x_i)
  Set p_i = x_i, g = argmax(scores), g_w = argmin(scores)

For iteration = 1 to K:

  Step 1: For each particle i:
    Sample r_v, r_p, r_g, r_w ~ Uniform(0, 1)
    C = r_v*φ_v + r_p*φ_p + r_g*φ_g + r_w*φ_w
    v_i = (1/C) * [r_v*φ_v*v_i + r_p*φ_p*(p_i - x_i)
                    + r_g*φ_g*(g - x_i) - r_w*φ_w*(g_w - x_i)]

  Step 2: For each particle i:
    x_i = x_i + λ * v_i
    score_i = f(x_i)
    If score_i > best_score(p_i): update p_i
    If score_i > best_score(g): update g
    If score_i < best_score(g_w): update g_w

    If particle i stagnated for restart_patience * patience:
      x_i = p_i; v_i = 0  (restart)

  Step 3: Check convergence
    If g unchanged for patience iterations: STOP
    λ = max(λ * φ_λ, λ_min)

Output: global best particle g
```

## 4.8 Utility Functions: Defining "Good"

The utility function is the objective that PSO optimizes. The paper uses four types:

### Single Task: Validation Accuracy
Simply the fraction of correct answers on a validation set.

### Multi-Task: Harmonic Mean
$$\text{utility} = \frac{2 \cdot s_1 \cdot s_2}{s_1 + s_2}$$

The harmonic mean penalizes imbalanced performance — a model scoring 90% on task 1 and 10% on task 2 gets 18%, not 50%.

### Reward Model: Average RM Score
Generate responses to prompts, score with a reward model.

### Human Interest: LLM-as-Judge
Generate responses, have Gemini rate them 1-10.

The beauty is that **changing the utility function is all you need** to adapt the algorithm to different objectives.

## 4.9 Exploring the Harmonic Mean

Let's understand why the harmonic mean is better than the arithmetic mean for multi-task optimization:

In [ ]:
# Compare harmonic and arithmetic means for different score distributions
scenarios = [
    (0.9, 0.9, "Both high"),
    (0.9, 0.1, "Very imbalanced"),
    (0.5, 0.5, "Both medium"),
    (0.7, 0.3, "Moderately imbalanced"),
    (0.95, 0.05, "Extremely imbalanced"),
]

print(f"{'Scenario':<25} {'Scores':>12} {'Arithmetic':>12} {'Harmonic':>10} {'Ratio H/A':>10}")
print("-" * 72)

for s1, s2, label in scenarios:
    arithmetic = (s1 + s2) / 2
    harmonic = 2 * s1 * s2 / (s1 + s2)
    print(f"{label:<25} ({s1:.2f}, {s2:.2f})  {arithmetic:10.3f}   {harmonic:10.3f}  {harmonic/arithmetic:10.3f}")

The harmonic mean strongly penalizes imbalance. When scores are (0.95, 0.05), the arithmetic mean is 0.50 but the harmonic mean is only 0.095. This forces the optimizer to find models that are balanced across both tasks.

## 4.10 Worked Example: One Complete Iteration

Let's trace through one iteration with concrete numbers.

**Setup**: 3 particles, 2D weight space, maximizing a function.

```
Iteration 5:
  Particle 0: position=[0.3, 0.7], velocity=[0.1, -0.05], personal_best=[0.4, 0.6], score=0.72
  Particle 1: position=[0.5, 0.2], velocity=[-0.1, 0.1], personal_best=[0.5, 0.3], score=0.68
  Particle 2: position=[0.8, 0.9], velocity=[0.05, 0.02], personal_best=[0.7, 0.8], score=0.81

  Global best: [0.7, 0.8] (score=0.81, from Particle 2)
  Global worst: [0.5, 0.2] (score=0.55, from iteration 3)

Hyperparameters: φ_v=0.2, φ_p=0.3, φ_g=0.4, φ_w=0.1, λ=0.85
```

In [ ]:
# Trace Particle 0's velocity update with specific random numbers
r_v, r_p, r_g, r_w = 0.73, 0.41, 0.89, 0.55
phi_v, phi_p, phi_g, phi_w = 0.2, 0.3, 0.4, 0.1

# Weighted components
iw = r_v * phi_v;  print(f"inertia_w  = {r_v} * {phi_v} = {iw:.3f}")
cw = r_p * phi_p;  print(f"cognitive_w = {r_p} * {phi_p} = {cw:.3f}")
sw = r_g * phi_g;  print(f"social_w   = {r_g} * {phi_g} = {sw:.3f}")
rw = r_w * phi_w;  print(f"repel_w    = {r_w} * {phi_w} = {rw:.3f}")
C = iw + cw + sw + rw
print(f"C = {iw:.3f} + {cw:.3f} + {sw:.3f} + {rw:.3f} = {C:.3f}")

# Normalize
iw_n, cw_n, sw_n, rw_n = iw/C, cw/C, sw/C, rw/C
print(f"\nNormalized weights: [{iw_n:.3f}, {cw_n:.3f}, {sw_n:.3f}, {rw_n:.3f}]")

In [ ]:
# Direction components
position = np.array([0.3, 0.7])
velocity = np.array([0.1, -0.05])
personal_best = np.array([0.4, 0.6])
global_best = np.array([0.7, 0.8])
global_worst = np.array([0.5, 0.2])

inertia_dir = velocity
cognitive_dir = personal_best - position
social_dir = global_best - position
repulsion_dir = -(global_worst - position)

print("Direction components:")
print(f"  inertia   = {inertia_dir}  (current velocity)")
print(f"  cognitive = {personal_best} - {position} = {cognitive_dir}  (toward personal best)")
print(f"  social    = {global_best} - {position} = {social_dir}  (toward global best)")
print(f"  repulsion = -({global_worst} - {position}) = {repulsion_dir}  (away from worst)")

# Combine
new_velocity = (iw_n * inertia_dir + cw_n * cognitive_dir 
                + sw_n * social_dir + rw_n * repulsion_dir)
print(f"\nNew velocity = {iw_n:.3f}*{inertia_dir} + {cw_n:.3f}*{cognitive_dir}")
print(f"             + {sw_n:.3f}*{social_dir} + {rw_n:.3f}*{repulsion_dir}")
print(f"             = [{new_velocity[0]:.4f}, {new_velocity[1]:.4f}]")

# Position update
step_length = 0.85
new_position = position + step_length * new_velocity
print(f"\nPosition update: {position} + {step_length} * [{new_velocity[0]:.3f}, {new_velocity[1]:.3f}]")
print(f"               = [{new_position[0]:.3f}, {new_position[1]:.3f}]")

## 4.11 Computational Cost Analysis

Each iteration requires:
- **6N merge operations** (4 for velocity directions + 1 velocity combination + 1 position update per particle)
- **N evaluations** (forward passes on validation set — the expensive part)

With N=20 and 200 validation examples:
- 120 merge operations (fast — tensor arithmetic on ~18M parameters)
- 20 model evaluations (requires loading model + running inference)

Typical search: 15-30 iterations → 300-600 total model evaluations.

### Dropout-K/N Acceleration
- **Dropout-K**: Skip entire iteration's evaluation with probability `dropK`
- **Dropout-N**: Skip individual particle evaluation with probability `dropN`
- With `dropK=0.5`, `dropN=0.5` → expected evaluations drop from 20 to ~5 per iteration (4x speedup).

## Exercise 4.1: Trace the Algorithm

Using the same setup from the worked example, complete the velocity and position updates for Particles 1 and 2. Use these random numbers:

- Particle 1: $r_v=0.22, r_p=0.67, r_g=0.35, r_w=0.91$
- Particle 2: $r_v=0.58, r_p=0.14, r_g=0.82, r_w=0.44$

In [ ]:
# TODO: Trace Particle 1
# position=[0.5, 0.2], velocity=[-0.1, 0.1], personal_best=[0.5, 0.3]
# global_best=[0.7, 0.8], global_worst=[0.5, 0.2], step_length=0.85
# r_v=0.22, r_p=0.67, r_g=0.35, r_w=0.91

# Your calculation here...

In [ ]:
# TODO: Trace Particle 2
# position=[0.8, 0.9], velocity=[0.05, 0.02], personal_best=[0.7, 0.8]
# global_best=[0.7, 0.8], global_worst=[0.5, 0.2], step_length=0.85
# r_v=0.58, r_p=0.14, r_g=0.82, r_w=0.44

# Your calculation here...

## Exercise 4.2: Utility Function Design

Design utility functions for these scenarios (write pseudocode):

1. **Code quality**: 10 code LLMs, 200 programming problems with test cases
2. **Multilingual translation**: English↔Spanish↔French, 100 pairs per direction
3. **Conflicting objectives**: Creative AND factual responses

*Your designs here:*

1. 
2. 
3. 

## Exercise 4.3: Sensitivity Predictions

Without running code, predict the effect of:

1. Setting `inertia=0.0` (no momentum)
2. Setting `social_coeff=0.0` (no social term)
3. Setting `repel_coeff=0.0` (no repulsion)
4. Setting `patience=1`
5. Setting `step_length_factor=1.0` (no decay)

*Your predictions here:*

1. 
2. 
3. 
4. 
5. 

---

**Next: [Module 5 — Code Architecture & Implementation Walkthrough](module_05_code_walkthrough.ipynb)**

---